In [1]:
from scripts.Utils import NER_Utils
import torch
from transformers import RobertaForTokenClassification, RobertaTokenizerFast, TrainingArguments, Trainer
from scripts.Reader import obtain_dataset, obtain_label_list

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Current Device: 0 NVIDIA GeForce RTX 4070 Ti


In [2]:
datasets, label_list, label2id, id2label = obtain_dataset("TempEval3", "BIO")

In [3]:
# Load tokenizer and model
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True)
model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
training_args = TrainingArguments(
    output_dir="./results/EventTimex-NER",
    logging_dir="./logs/EventTimex-NER",
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=100,
    num_train_epochs=10,
    save_total_limit=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [5]:
utils = NER_Utils(tokenizer, label_list)
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/3326 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

In [6]:
datasets

DatasetDict({
    train: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 22865
    })
    eval: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3326
    })
    test: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 223
    })
})

In [7]:
timex3_ner = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=utils.data_collator,
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

C:\Users\Harry\AppData\Local\Temp\ipykernel_15128\716834860.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  timex3_ner = Trainer(


In [8]:
timex3_ner.train()

Step,Training Loss,Validation Loss,Precision,Recall,F1
100,0.250300,0.128728,0.870351,0.825728,0.847452
200,0.098100,0.124248,0.872116,0.855146,0.863548
300,0.082100,0.124325,0.874871,0.849201,0.861845
400,0.083900,0.127929,0.835322,0.886341,0.860076
500,0.078600,0.118724,0.880165,0.856922,0.868388
600,0.076500,0.116485,0.869245,0.876226,0.872722
700,0.077500,0.108479,0.880492,0.868118,0.874261
800,0.074000,0.116949,0.861853,0.894062,0.877662
900,0.072900,0.122498,0.856120,0.895915,0.875566
1000,0.067800,0.124437,0.893364,0.840939,0.866359


TrainOutput(global_step=14300, training_loss=0.02595733968215389, metrics={'train_runtime': 4911.847, 'train_samples_per_second': 46.551, 'train_steps_per_second': 2.911, 'total_flos': 5.098817875533275e+16, 'train_loss': 0.02595733968215389, 'epoch': 10.0})

In [9]:
timex3_ner.evaluate(datasets["test"])

{'eval_loss': 0.2513830363750458,
 'eval_precision': 0.8468677494199536,
 'eval_recall': 0.8257918552036199,
 'eval_f1': 0.8361970217640321,
 'eval_runtime': 0.3682,
 'eval_samples_per_second': 605.659,
 'eval_steps_per_second': 19.012,
 'epoch': 10.0}

In [11]:
timex3_ner.save_model("./results/EventTimex-NER/final_model")

In [60]:
from scripts.Utils import NER_Utils
utils = NER_Utils(tokenizer, label_list)
encodings=tokenizer(list(datasets["test"]["tokens"]), padding=True, truncation=True, return_tensors="pt", is_split_into_words=True)
encodings.to(device)
with torch.no_grad():
    outputs = timex3_ner.model(**encodings)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

utils.classification_rep(datasets["test"]["label"], predictions, average='strict', encodings=encodings)


AttributeError: 'NER_Utils' object has no attribute 'classification_rep'